- Read bronze circuits table
- Keep only the columns required for analytics.
- standardise column names using snake_case.
- Concatenate name.givename and family name to create a new column called driver_name and transform the values to tile case.
- Filer out rows where circuit_id is null
- Remobe duplicate records.
- Transform values of columns circuit_name and locality to Title case
- write the trandformed data to silver circuits table

In [0]:
%run ../00.Common/01.environment-config


In [0]:
bronze_table = f"{catalog_nameone}.{bronze_schema}.drivers"
silver_table = f"{catalog_nameone}.{silver_schema}.drivers"

In [0]:
drivers_df = spark.read.table(bronze_table)

In [0]:
from pyspark.sql.functions import col
from pyspark.sql import functions as F

drivers_dropped_df = drivers_df.drop(F.col("url"))

In [0]:
drivers_renamed_df = (
  drivers_dropped_df
    .withColumnsRenamed({
      "driverId": "driver_id",
      "dateOfBirth": "date_Of_Birth"
    })
)

In [0]:
drivers_concanate_df = (
    drivers_renamed_df
    .withColumn("driver_name", 
                                  F.initcap(F.concat_ws(" ", F.col("name.givenName"), F.col("name.familyName"))))
    .drop("name")

    )

In [0]:
display(drivers_concanate_df)

In [0]:
drivers_distinct_df = drivers_concanate_df.dropDuplicates(["driver_id"])

In [0]:
drivers_final_df = (
    drivers_concanate_df
        .withColumn('nationality', F.initcap(F.col('nationality')))
)

In [0]:
drivers_final_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)

In [0]:
display(spark.table(silver_table))